# FEVS 2024 – Step 2: Data cleaning

**Goal:** turn the raw 2024 FEVS Public Release Data File into a clean, analysis-ready file.

What this notebook does:
1. Load the raw file and the codebook
2. Check the basics (rows, columns, duplicates, weights)
3. Turn "Do Not Know" / "No Basis to Judge" answers (`X`) into blanks
4. Convert every survey answer to a number and check it is in the expected range
5. Add readable labels for agencies, work units and demographics
6. Drop sensitive demographics we don't need (race, ethnicity, disability, sex)
7. Save the clean file and a question dictionary to `data/clean/`

The raw file is never changed.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw/2024")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

DATA_FILE = RAW / "2024_OPM_FEVS_PRDF_rv20261007.csv"
CODEBOOK = RAW / "2024_OPM_FEVS_PRDF_Codebook.xlsx"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Load the raw data and the codebook

In [2]:
raw = pd.read_csv(DATA_FILE, dtype=str)   # read everything as text first, convert on purpose later
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")

cb_items  = pd.read_excel(CODEBOOK, sheet_name="Survey Items")
cb_demo   = pd.read_excel(CODEBOOK, sheet_name="Demographics")
cb_agency = pd.read_excel(CODEBOOK, sheet_name="Agency Variable")
cb_level1 = pd.read_excel(CODEBOOK, sheet_name="LEVEL1 Variable")

raw.head(3)

Rows: 674,207   Columns: 96


,RandomID,POSTWT,agency,level1,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16_1,Q16_2,Q16_3,Q16_4,Q16_5,Q16_6,...,Q63,Q64,Q65,Q66,Q67,Q68,Q69,Q70,Q71,Q72,Q86,Q87,Q88,Q89,Q90,Q91,DRNO,DHISP,DDIS,DAGEGRP,DSUPER,DFEDTEN,DSEX,DMIL,DLEAVING
0,112970976817,1.5976951118,XX,XXZZ,3,4,4,4,1,NaN,5,4,4,4,4,4,5,4,4,1,1,1,1,0,0,...,3,3,2,2,4,4,2,4,4,4,4,5,5,5,5,2,B,B,B,B,B,B,A,B,D
1,194868625278,1.8283597661,XX,XXZZ,3,3,4,4,4,4,3,3,3,3,3,3,4,3,3,0,0,0,0,0,1,...,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,3,B,B,B,B,B,B,A,A,A
2,152966380283,1.8673976014,XX,XXZZ,2,1,3,3,3,2,4,1,2,2,3,2,3,3,5,0,1,0,0,0,0,...,X,2,1,1,1,1,1,1,1,1,2,3,1,4,4,3,B,B,B,B,A,A,A,A,A


## 2. Basic checks

In [3]:
checks = {
    "rows": len(raw),
    "unique RandomID": raw["RandomID"].nunique(),
    "duplicate rows": int(raw.duplicated().sum()),
    "missing weight (POSTWT)": int(raw["POSTWT"].isna().sum()),
    "agencies": raw["agency"].nunique(),
    "work units (level1)": raw["level1"].nunique(),
}
pd.Series(checks, name="value").to_frame()

,value
rows,674207
unique RandomID,674207
duplicate rows,0
missing weight (POSTWT),0
agencies,36
work units (level1),163


## 3. Build the question dictionary from the codebook

One row per question: its text and which OPM index it belongs to (if any).

In [4]:
questions = (cb_items
             .drop_duplicates("VARIABLE")[["VARIABLE", "ITEM TEXT", "INDEX", "PERFORMANCE DIMENSION"]]
             .rename(columns={"VARIABLE": "question", "ITEM TEXT": "text",
                              "INDEX": "opm_index", "PERFORMANCE DIMENSION": "dimension"}))

# Groups of columns
multi_select = [c for c in raw.columns if c.startswith("Q16_")]          # 0/1 tick boxes
telework     = ["Q91"]                                                   # category, not a scale
scale_items  = [q for q in questions["question"]
                if q.startswith("Q") and q not in multi_select + telework]  # 1–5 scales

print(f"Scale questions: {len(scale_items)}  |  multi-select: {len(multi_select)}  |  telework: {len(telework)}")
questions.head(10)

Scale questions: 76  |  multi-select: 6  |  telework: 1


,question,text,opm_index,dimension
0,Q1,I am given a real opportunity to improve my sk...,NaN,Performance Dimension: Employee-Focused - Empl...
5,Q2,I feel encouraged to come up with new and bett...,Employee Engagement Index - Intrinsic Work Exp...,NaN
10,Q3,My work gives me a feeling of personal accompl...,Employee Engagement Index - Intrinsic Work Exp...,NaN
15,Q4,I know what is expected of me on the job.,Employee Engagement Index - Intrinsic Work Exp...,NaN
20,Q5,My workload is reasonable.,NaN,Performance Dimension: Employee-Focused - Work...
25,Q6,My talents are used well in the workplace.,Employee Engagement Index - Intrinsic Work Exp...,NaN
30,Q7,I know how my work relates to the agency's goals.,Employee Engagement Index - Intrinsic Work Exp...,NaN
35,Q8,I can disclose a suspected violation of any la...,NaN,Performance Dimension: Foundations - Merit Pri...
41,Q9,I have enough information to do my job well.,NaN,Performance Dimension: Foundations - Performan...
46,Q10,I receive the training I need to do my job well.,NaN,Performance Dimension: Employee-Focused - Empl...


## 4. Recode "Do Not Know" (X) to blank and convert to numbers

In the raw file, `X` means *Do Not Know*, *No Basis to Judge* or *No recent hires*.
These are not opinions on the 1–5 scale, so they become blanks (NaN) and are left out of averages.
We count them first, so we know how much we removed.

In [5]:
df = raw.copy()

x_counts = (df[scale_items] == "X").sum().sort_values(ascending=False)
print(f"Total 'X' answers turned into blanks: {int(x_counts.sum()):,}")
print("Questions with the most 'Do Not Know' answers:")
display(x_counts.head(8).rename("X answers").to_frame()
        .join(questions.set_index("question")["text"]))

df[scale_items] = df[scale_items].replace("X", np.nan).apply(pd.to_numeric, errors="coerce")
df[multi_select + telework] = df[multi_select + telework].apply(pd.to_numeric, errors="coerce")
df["POSTWT"] = pd.to_numeric(df["POSTWT"], errors="coerce")

Total 'X' answers turned into blanks: 891,565
Questions with the most 'Do Not Know' answers:


,X answers,text
Q24,83549,"New hires in my work unit (i.e., hired in the ..."
Q17,71115,"In my work unit, differences in performance ar..."
Q39,39005,"My organization shares results (for example, t..."
Q58,36728,My organization's senior leaders maintain high...
Q47,36589,I believe the results of this survey will be u...
Q63,35403,Senior leaders demonstrate support for Work-Li...
Q45,31937,"In my organization, arbitrary action, personal..."
Q61,31266,"Overall, how good a job do you feel is being d..."


## 5. Check every value is in the expected range

In [6]:
bad_scale = int(((df[scale_items] < 1) | (df[scale_items] > 5)).sum().sum())
bad_multi = int((~df[multi_select].isin([0, 1]) & df[multi_select].notna()).sum().sum())
bad_tele  = int((~df["Q91"].isin([1, 2, 3, 4]) & df["Q91"].notna()).sum())
bad_wt    = int((df["POSTWT"] <= 0).sum())

pd.Series({"scale answers outside 1–5": bad_scale,
           "multi-select not 0/1": bad_multi,
           "telework not 1–4": bad_tele,
           "weights ≤ 0": bad_wt}, name="problems").to_frame()

,problems
scale answers outside 1–5,0
multi-select not 0/1,0
telework not 1–4,0
weights ≤ 0,0


## 6. Add readable labels

- Agency and work-unit names from the codebook
- Demographic codes (A, B, C…) turned into words
- Telework answer turned into words

In [7]:
agency_names = dict(zip(cb_agency["CODE"], cb_agency["NAME"]))
level1_names = dict(zip(cb_level1["CODE"], cb_level1["NAME"]))
df.insert(3, "agency_name", df["agency"].map(agency_names))
df.insert(5, "level1_name", df["level1"].map(level1_names))

KEEP_DEMO = ["DSUPER", "DFEDTEN", "DAGEGRP", "DMIL", "DLEAVING"]
for col in KEEP_DEMO:
    labels = (cb_demo[cb_demo["VARIABLE"] == col]
              .set_index("RESPONSE VALUE")["RESPONSE CHOICES"].str.strip().to_dict())
    df[col] = df[col].map(labels)

tele_labels = (cb_items[cb_items["VARIABLE"] == "Q91"]
               .assign(v=lambda d: pd.to_numeric(d["RESPONSE VALUE"], errors="coerce"))
               .set_index("v")["RESPONSE CHOICES"].to_dict())
df["Q91_telework"] = df["Q91"].map(tele_labels)

# Simple flag for later retention analysis
df["plans_to_leave"] = np.where(df["DLEAVING"].isna(), np.nan,
                                (df["DLEAVING"] != "No").astype(float))

print("Unmatched agency codes:", int(df["agency_name"].isna().sum()),
      "| unmatched work units:", int(df["level1_name"].isna().sum()))
df[["agency", "agency_name", "level1", "level1_name"] + KEEP_DEMO].head()

Unmatched agency codes: 0 | unmatched work units: 0


,agency,agency_name,level1,level1_name,DSUPER,DFEDTEN,DAGEGRP,DMIL,DLEAVING
0,XX,All Other Agencies,XXZZ,All Other Agencies,Supervisor/Manager/Executive,Eleven to 20 years,40 or Older,No Prior Military Service,"Yes, to take another job outside the Federal G..."
1,XX,All Other Agencies,XXZZ,All Other Agencies,Supervisor/Manager/Executive,Eleven to 20 years,40 or Older,Military Service,No
2,XX,All Other Agencies,XXZZ,All Other Agencies,Non-Supervisor/Team Leader,Ten years or fewer,40 or Older,Military Service,No
3,XX,All Other Agencies,XXZZ,All Other Agencies,Non-Supervisor/Team Leader,Ten years or fewer,40 or Older,Military Service,No
4,XX,All Other Agencies,XXZZ,All Other Agencies,Supervisor/Manager/Executive,More than 20 years,40 or Older,Military Service,No


## 7. Drop sensitive demographics we don't need

Race (`DRNO`), Hispanic origin (`DHISP`), disability (`DDIS`) and sex (`DSEX`) are not needed
to answer the business question, so they are left out of the clean file.

In [8]:
DROP = ["DRNO", "DHISP", "DDIS", "DSEX"]
df = df.drop(columns=DROP)
print(f"Columns now: {df.shape[1]}")

Columns now: 96


## 8. How complete is the data?

In [9]:
missing = (df[scale_items + KEEP_DEMO].isna().mean().mul(100).round(1)
           .sort_values(ascending=False).rename("% blank").to_frame())
missing.head(12)

,% blank
Q24,13.0
DAGEGRP,12.0
Q17,10.8
Q58,9.0
Q63,8.7
Q39,8.1
Q61,7.8
Q47,7.5
Q45,7.4
DMIL,6.8


## 9. Quick sanity check against OPM

A simple, weighted % positive (4 or 5) for one question. If cleaning worked, the numbers look sensible
(OPM reports results as weighted % positive).

In [10]:
def pct_positive(data, q):
    s = data[[q, "POSTWT"]].dropna()
    return 100 * np.average(s[q] >= 4, weights=s["POSTWT"])

for q in ["Q46", "Q54", "Q70", "Q5"]:
    txt = questions.set_index("question").loc[q, "text"]
    print(f"{q}: {pct_positive(df, q):5.1f}% positive  –  {txt}")

Q46:  69.2% positive  –  I recommend my organization as a good place to work.
Q54:  78.8% positive  –  Overall, how good a job do you feel is being done by your immediate supervisor?
Q70:  69.5% positive  –  Considering everything, how satisfied are you with your job?
Q5:  63.3% positive  –  My workload is reasonable.


## 10. Save the clean file and the question dictionary

In [12]:
out_data = CLEAN / "fevs_2024_clean.csv.gz"
out_dict = CLEAN / "fevs_2024_questions.csv"

df.to_csv(out_data, index=False, compression="gzip")
questions.to_csv(out_dict, index=False)

print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {out_data}")
print(f"Saved {len(questions)} questions → {out_dict}")

Saved 674,207 rows × 96 columns → ..\data\clean\fevs_2024_clean.csv.gz
Saved 83 questions → ..\data\clean\fevs_2024_questions.csv


## Summary

- Rows kept: all respondents (nothing deleted; blanks only where people answered *Do Not Know*)
- Survey answers are now numbers 1–5; tick boxes 0/1; telework labelled
- Agency and work-unit names added; sensitive demographics removed
- Next step (Step 3): build the indices – OPM's Engagement Index to check the method, then my well-being and workload index